# CNN: C 1s peak properties

Three convolutions find local peak patterns before dense layers predict the properties. Each model predicts peak presence, center, measured FWHM, area, and shape for five C 1s components.

## 1. Load SESSA spectra and labels

Set `data_path` to the generated NPZ on the training machine. Derive each present peak's FWHM from its saved SESSA component curve. Keep all noisy versions of a case in the same split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = Path("datasets/sessa_c1s_v1/dataset.npz")  # Change on the training machine if needed.
assert data_path.exists(), f"Set data_path to the SESSA dataset on your training machine: {data_path}"
data = np.load(data_path)
energy = data["energy_ev"].astype(np.float32)
names = data["components"].tolist()
base_centers = {"CC": 285.0, "CO": 286.5, "CARBONYL": 287.7,
                "OCO": 288.1, "COO": 289.1}
base = np.array([base_centers[name] for name in names], dtype=np.float32)
base_tensor = torch.tensor(base, device=device)
shape_names = ["Gaussian", "Lorentzian", "pseudo-Voigt"]

def prepare_spectrum(values):
    values = np.asarray(values, dtype=np.float32)
    left = values[..., :10].mean(axis=-1, keepdims=True)
    right = values[..., -10:].mean(axis=-1, keepdims=True)
    line = left + (right - left) * np.linspace(0, 1, values.shape[-1], dtype=np.float32)
    corrected = np.maximum(values - line, 0)
    return corrected / np.maximum(corrected.max(axis=-1, keepdims=True), 1e-6)

def measured_fwhm(curve):
    """Width at half maximum of a saved SESSA component curve."""
    top = int(np.argmax(curve))
    half = curve[top] / 2
    left = np.flatnonzero(curve[:top] < half)
    right = np.flatnonzero(curve[top + 1:] < half)
    if not len(left) or not len(right):
        raise ValueError("A component has no measurable FWHM on the energy grid")
    i = left[-1]
    j = top + 1 + right[0]
    x_left = np.interp(half, [curve[i], curve[i + 1]], [energy[i], energy[i + 1]])
    x_right = np.interp(half, [curve[j], curve[j - 1]], [energy[j], energy[j - 1]])
    return x_right - x_left

presence = data["presence"].astype(bool)
component_curves = data["component_relative"]
width = np.zeros(presence.shape, dtype=np.float32)
for case, component in np.argwhere(presence):
    width[case, component] = measured_fwhm(component_curves[case, component])

# Label columns: present, center (eV), FWHM (eV), area (relative intensity × eV), shape code.
labels = np.zeros((*presence.shape, 5), dtype=np.float32)
labels[..., 0] = presence
labels[..., 1] = np.where(presence, data["center_ev"], base[None, :])
labels[..., 2] = width
labels[..., 3] = data["component_area_window"]
labels[..., 4] = np.where(presence, data["shape_code"], 0)

noise_levels = data["peak_counts"]
n_noise = len(noise_levels)
spectra = prepare_spectrum(data["poisson_counts"]).reshape(-1, len(energy))
targets = np.repeat(labels, n_noise, axis=0)
splits = np.repeat(data["split"], n_noise)
case_indices = np.repeat(np.arange(len(data["case_id"])), n_noise)
noise_indices = np.tile(np.arange(n_noise), len(data["case_id"]))

def make_loader(split, shuffle=False):
    chosen = splits == split
    dataset = TensorDataset(torch.from_numpy(spectra[chosen]),
                            torch.from_numpy(targets[chosen]),
                            torch.from_numpy(case_indices[chosen]),
                            torch.from_numpy(noise_indices[chosen]))
    return DataLoader(dataset, batch_size=128, shuffle=shuffle)

train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("validation")
test_loader = make_loader("test")
print(f"SESSA cases: {len(data['case_id'])}; training scans: {len(train_loader.dataset)}")
print("Peak types:", names)


## 2. Define the CNN

The final layer gives seven values per component: one presence score, center, FWHM, area, and three shape scores.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 16, 7, stride=2, padding=3), nn.ReLU(),
            nn.Conv1d(16, 32, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(16), nn.Flatten(),
            nn.Linear(64 * 16, 64), nn.ReLU(), nn.Linear(64, 5 * 7))

    def forward(self, x):
        return self.network(x.unsqueeze(1))

model = CNN().to(device)

def peak_outputs(raw):
    """Turn 35 network values into five sets of peak properties."""
    raw = raw.reshape(-1, len(names), 7)
    return {
        "presence_logit": raw[..., 0],
        "center": base_tensor + 0.6 * torch.tanh(raw[..., 1]),
        "fwhm": 0.4 + 1.2 * torch.sigmoid(raw[..., 2]),
        "area": F.softplus(raw[..., 3]),
        "shape_logit": raw[..., 4:7],
    }

print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


## 3. Train

Compare every predicted property with its SESSA label. Save the model with the lowest validation loss.

In [ ]:
epochs = 12
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
checkpoint = Path("outputs/sessa_models/cnn.pt")
checkpoint.parent.mkdir(parents=True, exist_ok=True)
best_validation = float("inf")

def loss_on_batch(x, y):
    output = model(x)
    raw = output[0] if isinstance(output, tuple) else output
    pred = peak_outputs(raw)
    present = y[..., 0]
    mask = present > 0.5
    n_present = mask.sum().clamp(min=1)
    presence_loss = F.binary_cross_entropy_with_logits(pred["presence_logit"], present)
    center_loss = (((pred["center"] - y[..., 1]) / 0.5).square() * mask).sum() / n_present
    width_loss = (((pred["fwhm"] - y[..., 2]) / 1.0).square() * mask).sum() / n_present
    area_loss = (((pred["area"] - y[..., 3]) / 2.0).square() * mask).sum() / n_present
    shape_loss = F.cross_entropy(pred["shape_logit"][mask], y[..., 4][mask].long())
    loss = presence_loss + center_loss + width_loss + area_loss + 0.25 * shape_loss
    if isinstance(output, tuple):
        _, reconstruction, mean, log_variance = output
        reconstruction_loss = F.mse_loss(reconstruction, x)
        kl_loss = -0.5 * torch.mean(1 + log_variance - mean.square() - log_variance.exp())
        loss = loss + 0.1 * reconstruction_loss + 0.001 * kl_loss
    return loss

for epoch in range(1, epochs + 1):
    model.train()
    for x, y, _, _ in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_on_batch(x, y)
        loss.backward()
        optimizer.step()

    model.eval()
    validation_loss = 0.0
    with torch.no_grad():
        for x, y, _, _ in val_loader:
            validation_loss += loss_on_batch(x.to(device), y.to(device)).item() * len(x)
    validation_loss /= len(val_loader.dataset)
    if validation_loss < best_validation:
        best_validation = validation_loss
        torch.save(model.state_dict(), checkpoint)
    print(f"Epoch {epoch:2d}: validation loss = {validation_loss:.4f}")
print("Best model:", checkpoint)


## 4. Measure test accuracy

Use held-out SESSA cases. Report presence precision, recall and F1; center, FWHM and area errors; and shape accuracy.

In [ ]:
model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()
predicted, actual, test_cases, test_noise = [], [], [], []
with torch.no_grad():
    for x, y, case, noise in test_loader:
        output = model(x.to(device))
        raw = output[0] if isinstance(output, tuple) else output
        p = peak_outputs(raw)
        predicted.append(np.stack([
            torch.sigmoid(p["presence_logit"]).cpu().numpy(),
            p["center"].cpu().numpy(), p["fwhm"].cpu().numpy(),
            p["area"].cpu().numpy(),
            p["shape_logit"].argmax(dim=-1).cpu().numpy()], axis=-1))
        actual.append(y.numpy())
        test_cases.append(case.numpy())
        test_noise.append(noise.numpy())
predicted = np.concatenate(predicted)
actual = np.concatenate(actual)
test_cases = np.concatenate(test_cases)
test_noise = np.concatenate(test_noise)

def property_metrics(p, y):
    def one_row(estimate, truth, name):
        true_present = truth[:, 0] > 0.5
        pred_present = estimate[:, 0] >= 0.5
        tp = np.sum(true_present & pred_present)
        precision = tp / max(np.sum(pred_present), 1)
        recall = tp / max(np.sum(true_present), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        return {
            "component": name,
            "presence_precision": precision,
            "presence_recall": recall,
            "presence_F1": f1,
            "center_MAE_eV": np.mean(np.abs(estimate[true_present, 1] - truth[true_present, 1])),
            "FWHM_MAE_eV": np.mean(np.abs(estimate[true_present, 2] - truth[true_present, 2])),
            "area_MAE": np.mean(np.abs(estimate[true_present, 3] - truth[true_present, 3])),
            "shape_accuracy": np.mean(estimate[true_present, 4] == truth[true_present, 4]),
        }
    rows = [one_row(p[:, j], y[:, j], name) for j, name in enumerate(names)]
    rows.append(one_row(p.reshape(-1, 5), y.reshape(-1, 5), "ALL"))
    return pd.DataFrame(rows).set_index("component")

metrics = property_metrics(predicted, actual)
print("Held-out SESSA test metrics (property errors are for truly present peaks):")
print(metrics.round(3).to_string())
metrics.to_csv(checkpoint.with_name("cnn_test_metrics.csv"))


## 5. Plot best and worst cases

Reconstruct peaks from the true and predicted properties with the same simple profile formula. Rank test scans by reconstruction error and plot three best and three worst cases from distinct simulations.

In [ ]:
def peak_profile(center, fwhm, shape):
    """Unit-height Gaussian, Lorentzian, or 50:50 pseudo-Voigt profile."""
    distance = (energy - center) / fwhm
    gaussian = np.exp(-4 * np.log(2) * distance**2)
    lorentzian = 1 / (1 + 4 * distance**2)
    mix = [0.0, 1.0, 0.5][int(shape)]
    return (1 - mix) * gaussian + mix * lorentzian

def peak_curve(center, fwhm, area, shape):
    profile = peak_profile(center, fwhm, shape)
    return profile * area / max(np.trapz(profile, energy), 1e-9)

def reconstruct(properties, predicted_values=False):
    total = np.zeros_like(energy)
    for peak in properties:
        is_present = peak[0] >= 0.5 if predicted_values else peak[0] > 0.5
        if is_present:
            total += peak_curve(peak[1], peak[2], peak[3], peak[4])
    return total

scores = []
for p, y in zip(predicted, actual):
    truth = reconstruct(y)
    estimate = reconstruct(p, predicted_values=True)
    scores.append(np.trapz(np.abs(estimate - truth), energy) /
                  max(np.trapz(truth, energy), 1e-6))
scores = np.asarray(scores)
print(f"Relative reconstruction error: mean {scores.mean():.3f}, median {np.median(scores):.3f}")

def three_distinct(order):
    chosen, seen = [], set()
    for index in order:
        case = int(test_cases[index])
        if case not in seen:
            chosen.append(index)
            seen.add(case)
        if len(chosen) == 3:
            break
    return chosen

best = three_distinct(np.argsort(scores))
worst = three_distinct(np.argsort(scores)[::-1])
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, index, group in zip(axes.flat, best + worst, ["Best"] * 3 + ["Worst"] * 3):
    ax.plot(energy, reconstruct(actual[index]), label="True peak properties", color="black")
    ax.plot(energy, reconstruct(predicted[index], True), label="Predicted peak properties",
            color="tab:red", linestyle="--")
    case_name = data["case_id"][test_cases[index]]
    ax.set_title(f"{group}: {case_name}, {noise_levels[test_noise[index]]} counts\nerror {scores[index]:.3f}")
    ax.set_xlabel("Binding energy (eV)")
    ax.set_ylabel("Relative intensity")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
figure_path = checkpoint.with_name("cnn_best_worst.png")
fig.savefig(figure_path, dpi=150)
plt.show()
print("Saved:", figure_path)


## 6. Predict real measurements

Download the sucrose and malonic acid C 1s measurements from the [paper's Zenodo dataset](https://zenodo.org/records/4624072) and display their predicted peak properties. These real spectra have no matching SESSA labels, so accuracy is measured only on the held-out SESSA test cases.

In [ ]:
from urllib.parse import quote
from urllib.request import urlopen

real_files = {
    "Sucrose 0 mbar": "Sucrose_0mbar.xy",
    "Sucrose 0.2 mbar": "Sucrose_0_2mbar.xy",
    "Sucrose 1 mbar": "Sucrose_1mbar .xy",
    "Sucrose 5 mbar": "Sucrose_5mbar.xy",
    "Malonic acid 0.2 mbar": "MalonicAcid_0_2mbar.xy",
    "Malonic acid 1 mbar": "MalonicAcid_1mbar.xy",
}

def read_c1s(filename):
    url = "https://zenodo.org/records/4624072/files/" + quote(filename) + "?download=1"
    with urlopen(url, timeout=90) as response:
        lines = response.read().decode("utf-8", errors="replace").splitlines()
    in_c1s, photon_energy, points = False, None, []
    for line in lines:
        if line.startswith("# Region:"):
            if in_c1s:
                break
            in_c1s = "C 1s" in line
        elif in_c1s and line.startswith("# Excitation Energy:"):
            photon_energy = float(line.split(":", 1)[1])
        elif in_c1s and line and not line.startswith("#"):
            values = line.split()
            if len(values) == 2:
                points.append([float(values[0]), float(values[1])])
    if not points or photon_energy is None:
        raise ValueError(f"No C 1s spectrum found in {filename}")
    points = np.asarray(points)
    kinetic, inverse = np.unique(points[:, 0], return_inverse=True)
    counts = np.bincount(inverse, weights=points[:, 1]) / np.bincount(inverse)
    binding = photon_energy - kinetic
    order = np.argsort(binding)
    return np.interp(energy, binding[order], counts[order])

rows = []
model.eval()
for sample, filename in real_files.items():
    x = prepare_spectrum(read_c1s(filename)).astype(np.float32)
    with torch.no_grad():
        output = model(torch.from_numpy(x[None]).to(device))
        raw = output[0] if isinstance(output, tuple) else output
        p = peak_outputs(raw)
    probability = torch.sigmoid(p["presence_logit"]).cpu().numpy()[0]
    center = p["center"].cpu().numpy()[0]
    fwhm = p["fwhm"].cpu().numpy()[0]
    area = p["area"].cpu().numpy()[0]
    shape = p["shape_logit"].argmax(dim=-1).cpu().numpy()[0]
    for j, component in enumerate(names):
        rows.append({"sample": sample, "component": component,
                     "presence_probability": probability[j],
                     "present": int(probability[j] >= 0.5),
                     "center_ev": center[j], "fwhm_ev": fwhm[j],
                     "area_window": area[j], "shape": shape_names[shape[j]]})

real_peaks = pd.DataFrame(rows)
print(real_peaks.round(3).to_string(index=False))
